# 02 - Entrenar Modelo 1 (binario: sana vs enferma)

Transfer learning con EfficientNetB0. Dos fases: cabeza + fine-tuning.
Augmentacion con albumentations (rotacion, color jitter, distorsion elastica).
Mixed precision float16 para mayor velocidad en T4.


In [ ]:
!pip install -q tensorflow albumentations scikit-learn matplotlib

In [ ]:
import tensorflow as tf
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)
print("GPU:", tf.config.list_physical_devices("GPU"))

IMG_SIZE = (224, 224)
BATCH = 16
EPOCHS_P1 = 15
EPOCHS_P2 = 25
LR_P1 = 0.001
LR_P2 = 1e-05

DATA = Path("./splits/clasificacion_binaria")
OUT = Path("./outputs")
OUT.mkdir(exist_ok=True)


In [ ]:

import albumentations as A
import numpy as np
import os
from pathlib import Path
from PIL import Image
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.efficientnet import preprocess_input


TRAIN_AUG = A.Compose([
    A.Rotate(limit=45, border_mode=0, p=0.7),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.4),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.3),
    A.GaussNoise(var_limit=(5, 25), p=0.25),
    A.OneOf([
        A.ElasticTransform(alpha=60, sigma=12, p=1.0),
        A.GridDistortion(num_steps=5, distort_limit=0.2, p=1.0),
        A.OpticalDistortion(distort_limit=0.15, p=1.0),
    ], p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=0, p=0.4),
    A.CoarseDropout(max_holes=6, max_height=32, max_width=32, fill_value=0, p=0.2),
])

VAL_AUG = A.Compose([])


class LeafSequence(Sequence):
    def __init__(self, directory, img_size=(224, 224), batch_size=16,
                 augment=False, class_mode="binary", shuffle=True):
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment
        self.class_mode = class_mode
        self.shuffle = shuffle

        self.samples = []
        self.class_indices = {}
        classes = sorted(p.name for p in Path(directory).iterdir() if p.is_dir())
        for i, cls in enumerate(classes):
            self.class_indices[cls] = i
            for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
                for fp in (Path(directory) / cls).glob(ext):
                    self.samples.append((str(fp), i))

        self.n = len(self.samples)
        self.classes = np.array([s[1] for s in self.samples])
        if shuffle:
            self._shuffle()

    def _shuffle(self):
        idx = np.random.permutation(len(self.samples))
        self.samples = [self.samples[i] for i in idx]
        self.classes = self.classes[idx]

    def __len__(self):
        return max(1, self.n // self.batch_size)

    def __getitem__(self, i):
        batch = self.samples[i * self.batch_size:(i + 1) * self.batch_size]
        imgs, labels = [], []
        for fp, label in batch:
            img = np.array(Image.open(fp).convert("RGB").resize(self.img_size))
            if self.augment:
                img = TRAIN_AUG(image=img)["image"]
            else:
                img = VAL_AUG(image=img)["image"]
            img = preprocess_input(img.astype(np.float32))
            imgs.append(img)
            labels.append(label)
        X = np.stack(imgs)
        if self.class_mode == "binary":
            Y = np.array(labels, dtype=np.float32)
        else:
            n_cls = len(self.class_indices)
            Y = np.eye(n_cls)[labels]
        return X, Y

    def on_epoch_end(self):
        if self.shuffle:
            self._shuffle()

train_gen = LeafSequence(
    DATA / "train", img_size=IMG_SIZE, batch_size=BATCH,
    augment=True, class_mode="binary", shuffle=True,
)
val_gen = LeafSequence(
    DATA / "val", img_size=IMG_SIZE, batch_size=BATCH,
    augment=False, class_mode="binary", shuffle=False,
)
print(f"Train: {train_gen.n} imagenes | Val: {val_gen.n} imagenes")
print("Clases:", train_gen.class_indices)
with open(OUT / "class_indices_model1_binary.json", "w") as f:
    json.dump(train_gen.class_indices, f)


In [ ]:

def construir(num_clases=1):
    base = tf.keras.applications.EfficientNetB0(
        weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3)
    )
    base.trainable = False
    x = base.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(
        256, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    )(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    if num_clases == 1:
        out = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    else:
        out = tf.keras.layers.Dense(num_clases, activation="softmax")(x)
    return tf.keras.Model(base.input, out, name="model1_binary")


model = construir()
loss = "binary_crossentropy" if 1 == 1 else "categorical_crossentropy"
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_P1),
    loss=loss,
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="prec"),
        tf.keras.metrics.Recall(name="rec"),
    ],
)
model.summary()


In [ ]:
cbs_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(OUT / "model1_binary_p1_best.keras"),
        monitor="val_accuracy", save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
]

print("=== FASE 1: cabeza (base congelada) ===")
h1 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_P1, callbacks=cbs_p1, verbose=1,
)


In [ ]:
for layer in model.layers:
    layer.trainable = True

for layer in model.layers[:-30]:
    layer.trainable = False

for layer in model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainables = sum(1 for l in model.layers if l.trainable)
print(f"Layers entrenables: {trainables}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_P2),
    loss=loss,
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="prec"),
        tf.keras.metrics.Recall(name="rec"),
    ],
)

cbs_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(OUT / "model1_binary_p2_best.keras"),
        monitor="val_accuracy", save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.3, patience=4, min_lr=1e-8, verbose=1
    ),
]

print("=== FASE 2: fine-tuning (ultimos 30 layers) ===")
h2 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_P2, initial_epoch=len(h1.history["loss"]),
    callbacks=cbs_p2, verbose=1,
)

model.save(OUT / "model1_binary.keras")
print("Modelo guardado en", OUT / "model1_binary.keras")


In [ ]:
acc = h1.history["accuracy"] + h2.history["accuracy"]
vacc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
loss_h = h1.history["loss"] + h2.history["loss"]
vloss = h1.history["val_loss"] + h2.history["val_loss"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
div = len(h1.history["accuracy"])
ep = range(1, len(acc) + 1)
for ax, t, v, ylabel in [(ax1, acc, vacc, "Accuracy"), (ax2, loss_h, vloss, "Loss")]:
    ax.plot(ep, t, "b-", label="train")
    ax.plot(ep, v, "r-", label="val")
    ax.axvline(div, color="gray", linestyle="--", label="inicio fine-tune")
    ax.set_xlabel("Epoca"); ax.set_ylabel(ylabel); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "model1_binary_curves.png", dpi=120)
plt.show()
